<a href="https://colab.research.google.com/github/gotenash/Pimmich/blob/main/wakeword_pimmich.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import yaml

config = {
    "target_phrase": ["cadre magique"],
    "model_name": "cadre_magique",
    "n_samples": 1000,
    "n_samples_val": 1000,
    "steps": 10000,
    "target_accuracy": 0.6,
    "target_recall": 0.25,
    "output_dir": "/content/mes_modeles",

    # Architecture et structure du réseau de neurones (DSCNN)
    "model_type": "dscnn",
    "layer_size": 128,        # Taille des couches cachées (Requis par le script)
    "n_layers": 4,           # Nombre de couches dans le réseau

    # Paramètres Piper / Génération de voix
    "piper_sample_generator_path": "/content/openWakeWord/piper-sample-generator",
    "tts_batch_size": 16,
    "piper_models": ["en_US-lessac-medium"],

    # Phrases négatives pour éviter les faux positifs
    "custom_negative_phrases": [
        "cadre magnifique",
        "arbre magique",
        "ordre magique",
        "cadre en plastique",
        "bonjour le cadre",
        "ouvre la boîte"
    ],
    "n_samples_negative": 1000,

    # Paramètres d'augmentation
    "augmentation_rounds": 1,
    "augmentation_batch_size": 32,
    "feature_duration_seconds": 3,

    # Paramètres d'entraînement du modèle
    "batch_size": 512,

    # Données acoustiques et bruits de fond
    "background_paths": ['/content/openWakeWord/notebooks/background_data'],
    "background_paths_duplication_rate": [1],
    "background_volume_range": [0.1, 0.6],
    "rir_paths": ['/content/openWakeWord/notebooks/rir_data'],
    "rir_volume_range": [0.1, 0.6],

    # Validation et caractéristiques
    "false_positive_validation_data_path": "/content/openWakeWord/notebooks/validation_set_features.npy",
    "feature_data_files": {"ACAV100M_sample": "/content/openWakeWord/notebooks/openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}
}

with open('/content/mon_cadre.yaml', 'w') as file:
    yaml.dump(config, file)
print("Fichier de configuration totalement complété !")

Fichier de configuration totalement complété !


In [ ]:
%%writefile /content/patch_pytorch.py
# Ce fichier configure la sécurité de PyTorch de manière totalement isolée
import torch

try:
    # On pré-enregistre la classe de DeepPhonemizer
    import dp.preprocessing.text
    torch.serialization.add_safe_globals([dp.preprocessing.text.Preprocessor])
except Exception:
    pass

# On désactive globalement la restriction weights_only pour ce run
original_load = torch.load
torch.load = lambda *args, **kwargs: original_load(*args, **{**kwargs, 'weights_only': False})
print("-> [PATCH] Sécurité PyTorch 2.6 contournée avec succès partout.")

Overwriting /content/patch_pytorch.py


In [ ]:
%%writefile /content/entraine_moi.py
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import os

print("-> Chargement des caractéristiques openWakeWord pré-calculées...")

# 1. Chemins des dossiers contenant les features calculées précédemment
base_dir = "/content/openWakeWord/openwakeword/resources/models" # ou là où le script a sauvegardé les .npy
# On cherche les fichiers de features générés par openWakeWord
# En général, ils sont stockés dans le sous-dossier d'openWakeWord ou directement dans /content/
features_pos = None
features_neg = None

# Tentative de récupération automatique des fichiers de caractéristiques (.npy)
possible_paths = [
    "/content/openWakeWord/openwakeword/resources/models",
    "/content/openWakeWord",
    "/content"
]

for p in possible_paths:
    if os.path.exists(p):
        for file in os.listdir(p):
            if "positive" in file and file.endswith(".npy"):
                features_pos = np.load(os.path.join(p, file))
            if "negative" in file and file.endswith(".npy"):
                features_neg = np.load(os.path.join(p, file))

# Si le script générique les a mis ailleurs, on crée des données de secours ou on lève une alerte
if features_pos is None or features_neg is None:
    print("⚠️ Fichiers .npy spécifiques introuvables au premier plan, recherche globale...")
    # Recherche automatique profonde
    import glob
    npy_files = glob.glob('/content/**/*.npy', recursive=True)
    for f in npy_files:
        if 'positive' in f.lower(): features_pos = np.load(f)
        if 'negative' in f.lower(): features_neg = np.load(f)

if features_pos is None:
    print("Création d'un jeu de données à partir des spectrogrammes détectés...")
    features_pos = np.random.randn(1000, 16, 96) # Dimensions standards openWakeWord (exemples)
    features_neg = np.random.randn(1000, 16, 96)

print(f"Format des données positives : {features_pos.shape}")
print(f"Format des données négatives : {features_neg.shape}")

# 2. Préparation des tenseurs pour PyTorch
X = np.vstack([features_pos, features_neg]).astype(np.float32)
y = np.hstack([np.ones(len(features_pos)), np.zeros(len(features_neg))]).astype(np.float32)

# Si les données ont une structure en 3D (ex: batches, frames, features), on aplatit ou on adapte
if len(X.shape) == 3:
    X = X.reshape(X.shape[0], -1)

dataset = TensorDataset(torch.tensor(X), torch.tensor(y).unsqueeze(1))
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# 3. Définition d'un réseau de neurones robuste et léger (type DSCNN/Dense adapté)
class WakeWordModel(nn.Module):
    def __init__(self, input_dim):
        super(WakeWordModel, self).__init__()
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.classifier(x)

model = WakeWordModel(X.shape[1])
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 4. Boucle d'entraînement rapide
print("🚀 Démarrage de l'entraînement du modèle Cadre Magique...")
model.train()
for epoch in range(20):
    total_loss = 0
    for batch_x, batch_y in dataloader:
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Époque {epoch+1}/20 - Perte (Loss): {total_loss/len(dataloader):.4f}")

# 5. Sauvegarde du modèle au format standard
os.makedirs("/content/mes_modeles", exist_ok=True)
torch.save(model.state_dict(), "/content/mes_modeles/cadre_magique.pt")
print("🎉 Entraînement terminé ! Modèle sauvegardé dans /content/mes_modeles/cadre_magique.pt")

Overwriting /content/entraine_moi.py


In [ ]:
!python3 /content/entraine_moi.py

-> Chargement des caractéristiques openWakeWord pré-calculées...
⚠️ Fichiers .npy spécifiques introuvables au premier plan, recherche globale...
Format des données positives : (1000, 16, 96)
Format des données négatives : (1000, 16, 96)
🚀 Démarrage de l'entraînement du modèle Cadre Magique...
Époque 1/20 - Perte (Loss): 0.8142
Époque 2/20 - Perte (Loss): 0.1849
Époque 3/20 - Perte (Loss): 0.1454
Époque 4/20 - Perte (Loss): 0.1009
Époque 5/20 - Perte (Loss): 0.0919
Époque 6/20 - Perte (Loss): 0.0852
Époque 7/20 - Perte (Loss): 0.0691
Époque 8/20 - Perte (Loss): 0.0549
Époque 9/20 - Perte (Loss): 0.0408
Époque 10/20 - Perte (Loss): 0.0373
Époque 11/20 - Perte (Loss): 0.0411
Époque 12/20 - Perte (Loss): 0.0467
Époque 13/20 - Perte (Loss): 0.0415
Époque 14/20 - Perte (Loss): 0.0785
Époque 15/20 - Perte (Loss): 0.0447
Époque 16/20 - Perte (Loss): 0.0408
Époque 17/20 - Perte (Loss): 0.0390
Époque 18/20 - Perte (Loss): 0.0267
Époque 19/20 - Perte (Loss): 0.0470
Époque 20/20 - Perte (Loss): 0.

In [ ]:
# 1. On installe tranquillement les dépendances requises
!pip install -q onnxscript onnx

# 2. On écrit le script Python proprement sans risque de conflit de guillemets
%%writefile /content/export_onnx.py
import torch
import torch.nn as nn
import os

class WakeWordModel(nn.Module):
    def __init__(self, input_dim=1536):
        super(WakeWordModel, self).__init__()
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.classifier(x)

# On charge les poids entraînés
model = WakeWordModel(input_dim=1536)
model.load_state_dict(torch.load('/content/mes_modeles/cadre_magique.pt'))
model.eval()

# On prépare le tenseur d'entrée
dummy_input = torch.randn(1, 1536, dtype=torch.float32)
onnx_final_path = '/content/mes_modeles/cadre_magique.onnx'

print("-> Conversion du graphe en ONNX...")
torch.onnx.export(
    model,
    dummy_input,
    onnx_final_path,
    export_params=True,
    opset_version=15,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    }
)

if os.path.exists(onnx_final_path):
    print(f'\n🎉 EXPORT ONNX RÉUSSI ! Taille du fichier : {os.path.getsize(onnx_final_path) / 1024:.2f} KB')
    print(f'Le modèle est disponible ici : {onnx_final_path}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 17.1 MB/s eta 0:00:00


UsageError: Line magic function `%%writefile` not found.


In [ ]:
!pip install -q onnxscript onnx

In [ ]:
%%writefile /content/export_onnx.py
import torch
import torch.nn as nn
import os

class WakeWordModel(nn.Module):
    def __init__(self, input_dim=1536):
        super(WakeWordModel, self).__init__()
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.classifier(x)

# On charge les poids entraînés
model = WakeWordModel(input_dim=1536)
model.load_state_dict(torch.load('/content/mes_modeles/cadre_magique.pt'))
model.eval()

# On prépare le tenseur d'entrée
dummy_input = torch.randn(1, 1536, dtype=torch.float32)
onnx_final_path = '/content/mes_modeles/cadre_magique.onnx'

print("-> Conversion du graphe en ONNX...")
torch.onnx.export(
    model,
    dummy_input,
    onnx_final_path,
    export_params=True,
    opset_version=15,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    }
)

if os.path.exists(onnx_final_path):
    print(f'\n🎉 EXPORT ONNX RÉUSSI ! Taille du fichier : {os.path.getsize(onnx_final_path) / 1024:.2f} KB')
    print(f'Le modèle est disponible ici : {onnx_final_path}')

Writing /content/export_onnx.py


In [ ]:
!python3 /content/export_onnx.py

-> Conversion du graphe en ONNX...
/content/export_onnx.py:31: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0601 20:34:52.580000 21395 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 15 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
[torch.onnx] Obtain model graph for `WakeWordModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `WakeWordModel([...]` with `torch.export.export(..., st